# Module 6 Homework

## Question 1
- Install Spark and PySpark
- Install PySpark follow this guide:

    - Install Spark
    - Run PySpark
    - Create a local spark session
Execute spark.version. What's the output?

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework') \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 07:51:24 WARN Utils: Your hostname, codespaces-b285a2, resolves to a loopback address: 127.0.0.1; using 10.0.3.64 instead (on interface eth0)
26/03/09 07:51:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 07:51:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


## Question 2: Yellow November 2025
- Read the November 2025 Yellow into a Spark Dataframe.

- Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)?

In [4]:
!mkdir -p data/homework/raw
!mkdir -p data/homework/parquet

!wget -nc https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet \
    -O data/homework/raw/yellow_tripdata_2025-11.parquet

--2026-03-09 07:54:30--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.4, 18.239.115.146, 18.239.115.86, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘data/homework/raw/yellow_tripdata_2025-11.parquet’

data/homework/raw/y 100%[===================>]  67.84M   227MB/s    in 0.3s    

2026-03-09 07:54:31 (227 MB/s) - ‘data/homework/raw/yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [9]:
# Read the November 2025 Yellow into a Spark Dataframe.
df_trips = spark.read.parquet('data/homework/raw/yellow_tripdata_2025-11.parquet')

In [7]:
!ls -lh data/homework/raw

total 68M
-rw-rw-rw- 1 codespace codespace 68M Dec 19 15:51 yellow_tripdata_2025-11.parquet


In [11]:
# Repartition the Dataframe to 4 partitions and save it to parquet.
df_trips.repartition(4).write.parquet('data/homework/parquet', mode='overwrite')

In [13]:
# What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)?
!ls -lh data/homework/parquet/*.parquet

-rw-r--r-- 1 codespace codespace 26M Mar  9 08:01 data/homework/parquet/part-00000-71ca88a9-ba8b-4ecf-b61c-b5b57f5502b0-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar  9 08:01 data/homework/parquet/part-00001-71ca88a9-ba8b-4ecf-b61c-b5b57f5502b0-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar  9 08:01 data/homework/parquet/part-00002-71ca88a9-ba8b-4ecf-b61c-b5b57f5502b0-c000.snappy.parquet
-rw-r--r-- 1 codespace codespace 26M Mar  9 08:01 data/homework/parquet/part-00003-71ca88a9-ba8b-4ecf-b61c-b5b57f5502b0-c000.snappy.parquet


Select the answer which most closely matches.

- 6MB
- 25MB
- 75MB
- 100MB
The average size is **25MB.**

## Question 3: Count records
How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

In [14]:
from pyspark.sql import functions as F

df_trips = df_trips.withColumn('pickup_date', F.to_date(df_trips.tpep_pickup_datetime))
df_trips.filter(df_trips.pickup_date == '2025-11-15').count()

162604

On November 15th, a total of **162,604** trips were started.

## Question 4:Longest trip
What is the length of the longest trip in the dataset in hours?

- 22.7
- 58.2
- 90.6
- 134.5

In [15]:
from pyspark.sql import types
from datetime import datetime

def duration(start: datetime, end: datetime) -> int:
    # Difference in minutes (absolute value to support inverted values)
    minutes = abs((end - start).total_seconds()) / 60

    # Conversion to hours
    hours = minutes / 60

    return hours

duration_udf = F.udf(duration, returnType=types.FloatType())

df_trips \
    .withColumn('duration_hours', duration_udf(df_trips.tpep_pickup_datetime, df_trips.tpep_dropoff_datetime)) \
    .groupBy() \
    .max('duration_hours') \
    .show()

/workspaces/Data-Engineering-ZoomCamp/06.batch/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)
[Stage 12:=============================>                            (1 + 1) / 2]

+-------------------+
|max(duration_hours)|
+-------------------+
|           90.64667|
+-------------------+



The length of the longest trip in the dataset in hours is **90.6**

## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

It runs on port **4040**.

In [16]:
!wget -nc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv \
    -O data/homework/raw/taxi_zone_lookup.csv

--2026-03-09 08:04:17--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.4, 18.239.115.213, 18.239.115.146, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘data/homework/raw/taxi_zone_lookup.csv’

data/homework/raw/t 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 08:04:17 (796 MB/s) - ‘data/homework/raw/taxi_zone_lookup.csv’ saved [12331/12331]



In [23]:
df_zones = spark.read.csv('data/homework/raw/taxi_zone_lookup.csv', header=True)

df_trips.createOrReplaceTempView('trips')
df_zones.createOrReplaceTempView('zones')

spark.sql("""
WITH trips_by_pickup_location AS (
    SELECT PULocationID, COUNT(*) AS trip_count
    FROM trips
    GROUP BY PULocationID
)
SELECT 
    z.Zone, 
    SUM(COALESCE(t.trip_count, 0)) AS trip_count
FROM zones AS z
LEFT JOIN trips_by_pickup_location AS t 
    ON t.PULocationID = z.LocationID
WHERE z.Zone IN (
    'Governor''s Island/Ellis Island/Liberty Island',
    'Arden Heights',
    'Rikers Island',
    'Jamaica Bay'
)
GROUP BY z.Zone
ORDER BY trip_count
""").show(truncate=False)

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Governor's Island/Ellis Island/Liberty Island|1         |
|Arden Heights                                |1         |
|Rikers Island                                |4         |
|Jamaica Bay                                  |5         |
+---------------------------------------------+----------+



## Question 6: Least Frequent Pickup Location Zone

Load the zone lookup data into a temp view in Spark.
Using the zone lookup data and the Yellow November 2025 dataset, the analysis restricted to the four zones in the question produced the following trip counts:

- Governor's Island/Ellis Island/Liberty Island → 1 trip  
- Arden Heights → 1 trip  
- Rikers Island → 4 trips  
- Jamaica Bay → 5 trips  

**Final Answer:**  
The least frequent pickup zones among the given options are **Governor's Island/Ellis Island/Liberty Island** and **Arden Heights**, each with **only 1 trip recorded**.


In [25]:
spark.stop()

In [26]:
def cleanup_data(path="/workspaces/Data-Engineering-ZoomCamp/06.batch/homework/data"):
    import shutil, os
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Deleted {path}")
    else:
        print(f"{path} not found")

In [27]:
cleanup_data()

Deleted /workspaces/Data-Engineering-ZoomCamp/06.batch/homework/data
